In [ ]:
""" 
由于原代码使用了 TensorFlow 1.x 的 相关代码，不适用仓库中要求的 TensorFlow 2.x
因此我将代码重写为 TensorFlow 2.x 的版本，使用 Keras API 来构建和训练模型，达到了相同的效果。
"""
import os
import numpy as np
import tensorflow as tf

def _read_idx_images(path):
    with open(path, 'rb') as f:
        data = f.read()
    num_images = int.from_bytes(data[4:8], byteorder='big')
    rows = int.from_bytes(data[8:12], byteorder='big')
    cols = int.from_bytes(data[12:16], byteorder='big')
    images = np.frombuffer(data, dtype=np.uint8, offset=16)
    images = images.reshape(num_images, rows * cols).astype(np.float32)
    return images

def _read_idx_labels(path):
    with open(path, 'rb') as f:
        data = f.read()
    num_labels = int.from_bytes(data[4:8], byteorder='big')
    labels = np.frombuffer(data, dtype=np.uint8, offset=8)
    labels = labels.reshape(num_labels)
    one_hot = np.eye(10, dtype=np.float32)[labels]
    return one_hot

class DataSet(object):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
        self.num_examples = images.shape[0]
        self._index_in_epoch = 0
        self._perm = np.arange(self.num_examples)
        np.random.shuffle(self._perm)

    def next_batch(self, batch_size):
        start = self._index_in_epoch
        end = start + batch_size
        if end <= self.num_examples:
            batch_idx = self._perm[start:end]
            self._index_in_epoch = end
        else:
            rest_idx = self._perm[start:self.num_examples]
            np.random.shuffle(self._perm)
            new_end = end - self.num_examples
            new_idx = self._perm[0:new_end]
            batch_idx = np.concatenate([rest_idx, new_idx], axis=0)
            self._index_in_epoch = new_end
        return self.images[batch_idx], self.labels[batch_idx]

class MNISTData(object):
    def __init__(self, train_images, train_labels, test_images, test_labels):
        self.train = DataSet(train_images, train_labels)
        self.test = DataSet(test_images, test_labels)

raw_dir = os.path.join('.', 'mnist', 'MNIST', 'raw')
train_images = _read_idx_images(os.path.join(raw_dir, 'train-images-idx3-ubyte'))
train_labels = _read_idx_labels(os.path.join(raw_dir, 'train-labels-idx1-ubyte'))
test_images = _read_idx_images(os.path.join(raw_dir, 't10k-images-idx3-ubyte'))
test_labels = _read_idx_labels(os.path.join(raw_dir, 't10k-labels-idx1-ubyte'))
mnist = MNISTData(train_images, train_labels, test_images, test_labels)

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 2000
batch_size = 100

# TF2: 直接使用 Keras 构建模型（保持与原网络近似的结构）
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, kernel_size=(7, 7), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2, padding='same'),
    tf.keras.layers.Conv2D(64, kernel_size=(5, 5), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2, padding='same'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dropout(rate=1.0 - keep_prob_rate),
    tf.keras.layers.Dense(10, activation='softmax'),
])

optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
 )

# 归一化并调整输入形状
mnist.train.images = (mnist.train.images / 255.0).reshape(-1, 28, 28, 1)
mnist.test.images = (mnist.test.images / 255.0).reshape(-1, 28, 28, 1)

for i in range(max_epoch):
    batch_xs, batch_ys = mnist.train.next_batch(batch_size)
    model.train_on_batch(batch_xs, batch_ys)

    if i % 100 == 0:
        _, test_acc = model.evaluate(
            mnist.test.images[:1000],
            mnist.test.labels[:1000],
            verbose=0
        )
        print(test_acc)

0.23399999737739563
0.8740000128746033
0.9139999747276306
0.9330000281333923
0.9430000185966492
0.9580000042915344
0.9670000076293945
0.9760000109672546
0.9750000238418579
0.9769999980926514
0.9810000061988831
0.9800000190734863
0.9800000190734863
0.9810000061988831
0.9810000061988831
0.984000027179718
0.984000027179718
0.9850000143051147
0.984000027179718
0.984000027179718
